# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SymbolPamnani/Flyrank-ML-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Answer

I chose Logistic Regression as the first learned model for this ranking task.

The goal is to rank content items for review, not to produce a final yes/no decision. Logistic Regression provides a probability score for each item, which can be used to rank the content and evaluate the top-K items with Precision@50.

I chose it as the first model because it is relatively simple and interpretable. The model can show which features are associated with higher or lower predicted decline-proxy scores.

The target is the defined decline proxy: `trend_direction == "down"`. This is a dataset-defined proxy rather than an independently observed future outcome, so the model should be interpreted as decision support for prioritization rather than as proof that a page will decline.

The primary metric is Precision@50, matching the Week-4 baseline objective.

In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit

DATA_PATH = "content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

print("\nTarget distribution:")
target = (df["trend_direction"] == "down").astype(int)
print(target.value_counts())
print(f"\nDeclining-proxy base rate: {target.mean():.3f}")
print(f"Declining-proxy base rate (%): {target.mean() * 100:.1f}%")

Dataset shape: (30000, 44)

Target distribution:
trend_direction
1    16262
0    13738
Name: count, dtype: int64

Declining-proxy base rate: 0.542
Declining-proxy base rate (%): 54.2%


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## Answer

I use a grouped train/test split by `client_id`.

The reason is that multiple content items belong to the same client, so randomly splitting individual rows could place content from the same client in both training and test sets. A client-grouped split gives a stricter test of whether the learned ranking generalizes to clients that were not used for training.

I use 80% of the clients for training and 20% for the holdout set, with `random_state=42` for reproducibility.

The same holdout set is used for both the learned model and the Week-4 baseline so their Precision@50 values are directly comparable.

`client_id` is used only for grouping and is not included as a model feature.

In [2]:
# Create a client-grouped holdout split.
groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, target, groups=groups)
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

y_train = target.iloc[train_idx].values
y_test = target.iloc[test_idx].values

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nTrain clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

print("\nClient overlap:")
print(
    set(train_df["client_id"]).intersection(
        set(test_df["client_id"])
    )
)

print("\nTrain decline-proxy rate:", f"{y_train.mean():.3f}")
print("Test decline-proxy rate:", f"{y_test.mean():.3f}")

Train rows: 23837
Test rows: 6163

Train clients: 25
Test clients: 7

Client overlap:
set()

Train decline-proxy rate: 0.550
Test decline-proxy rate: 0.511


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 3. Answer

The learned model uses metadata available before the decline-proxy outcome is defined.

Numeric features include search demand, competition, content size, content age, and days since the last update. Categorical features describe competition level, content type, search intent, age tier, freshness tier, word-count tier, and character-count tier.

I exclude `trend_direction` and `trend_pct` because they define the decline proxy. I also exclude recent performance windows, 90-day performance aggregates, derived search-performance flags, and identifiers because they can overlap the outcome definition or act as identifiers rather than predictive features.

The baseline is recalculated on the same held-out test set using the same Week-4 scoring rule. This makes the model-versus-baseline comparison an apples-to-apples comparison.

On the held-out client split, the W04 rule baseline achieved Precision@50 of 0.700, while Logistic Regression achieved 0.380. The declining-proxy base rate was 0.511. In this evaluation, the learned model did not outperform the simple rule baseline or the base rate.

Precision@50 remains the primary metric because the practical decision is to prioritize a small review queue.

In [7]:
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
]

feature_columns = numeric_features + categorical_features

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print("\nExcluded from modeling:")
print([
    "trend_direction",
    "trend_pct",
    "recent performance windows",
    "90-day performance fields",
    "impression_tier",
    "position_tier",
    "provider_used",
    "model_used",
    "content_id",
    "client_id",
])

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median", add_indicator=True)
        ),
        (
            "scaler",
            StandardScaler()
        ),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        ),
    ]
)

X_train = train_df[feature_columns]
X_test = test_df[feature_columns]

model.fit(X_train, y_train)

model_scores = model.predict_proba(X_test)[:, 1]

print("Model trained successfully.")
print("Test scores:", len(model_scores))
print("Score range:", f"{model_scores.min():.3f} to {model_scores.max():.3f}")

def precision_at_k(scores, labels, k=50):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    top_k = np.argsort(-scores)[:k]

    return labels[top_k].mean()


def build_w04_baseline(test_data):
    baseline = test_data[
        [
            "content_id",
            "search_volume",
            "days_since_last_update",
            "content_age_days",
        ]
    ].copy()

    baseline["search_volume_rank"] = (
        baseline["search_volume"]
        .rank(pct=True)
        .fillna(0.5)
    )

    baseline["freshness_rank"] = (
        baseline["days_since_last_update"]
        .rank(pct=True)
    )

    baseline["age_rank"] = (
        baseline["content_age_days"]
        .rank(pct=True)
    )

    baseline["baseline_score"] = (
        0.50 * baseline["search_volume_rank"]
        + 0.30 * baseline["freshness_rank"]
        + 0.20 * baseline["age_rank"]
    )

    return baseline.sort_values(
        "baseline_score",
        ascending=False
    ).reset_index(drop=True)


baseline_test = build_w04_baseline(test_df)


# Align the decline-proxy labels with the sorted baseline queue
target_lookup = test_df[["content_id"]].copy()
target_lookup["declining_proxy"] = y_test

baseline_eval = baseline_test.merge(
    target_lookup,
    on="content_id",
    how="left",
    validate="one_to_one"
)

baseline_scores = baseline_eval["baseline_score"].values
baseline_labels = baseline_eval["declining_proxy"].values

baseline_p20 = precision_at_k(
    baseline_scores,
    baseline_labels,
    k=20
)

baseline_p50 = precision_at_k(
    baseline_scores,
    baseline_labels,
    k=50
)

base_rate = baseline_labels.mean()

print("\nCorrected evaluation:")
print(f"Declining-proxy base rate: {base_rate:.3f}")
print(f"W04 Rule Baseline Precision@20: {baseline_p20:.3f}")
print(f"W04 Rule Baseline Precision@50: {baseline_p50:.3f}")
print(f"Logistic Regression Precision@20: {model_p20:.3f}")
print(f"Logistic Regression Precision@50: {model_p50:.3f}")


comparison = pd.DataFrame(
    {
        "Method": [
            "Declining-proxy base rate",
            "W04 Rule Baseline",
            "Logistic Regression",
        ],
        "Precision@20": [
            base_rate_test,
            baseline_p20,
            model_p20,
        ],
        "Precision@50": [
            base_rate_test,
            baseline_p50,
            model_p50,
        ],
    }
)

print(comparison.to_string(index=False))

Numeric features:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update']

Categorical features:
['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier']

Excluded from modeling:
['trend_direction', 'trend_pct', 'recent performance windows', '90-day performance fields', 'impression_tier', 'position_tier', 'provider_used', 'model_used', 'content_id', 'client_id']
Model trained successfully.
Test scores: 6163
Score range: 0.174 to 0.789

Corrected evaluation:
Declining-proxy base rate: 0.511
W04 Rule Baseline Precision@20: 0.500
W04 Rule Baseline Precision@50: 0.400
Logistic Regression Precision@20: 0.300
Logistic Regression Precision@50: 0.380
                   Method  Precision@20  Precision@50
Declining-proxy base rate      0.510952      0.510952
        W04 Rule Baseline      0.500000      0.400000
      Logistic Regression      0.300000      0.380000


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Answer

I inspected the highest-ranked test items that were not part of the declining proxy.

These are ranking false positives: the model assigned them high priority, but their observed `trend_direction` was not `down`.

I also inspected the model coefficients to identify which transformed features contribute most strongly to the predicted decline-proxy score.

The feature interpretation is directional rather than causal. A positive coefficient means that the corresponding feature is associated with a higher predicted score within this fitted model; it does not show that the feature causes decline.

In [8]:
test_results = test_df[
    [
        "content_id",
        "client_id",
        "search_volume",
        "days_since_last_update",
        "content_age_days",
        "trend_direction",
    ]
].copy()

test_results["declining_proxy"] = y_test
test_results["model_score"] = model_scores

test_results = test_results.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

top50 = test_results.head(50).copy()

false_positives = top50[
    top50["declining_proxy"] == 0
].copy()

print("Top-50 model items:", len(top50))
print("Top-50 false positives:", len(false_positives))

print("\nFalse-positive rate within top 50:")
print(f"{len(false_positives) / 50:.3f}")

print("\nSample false positives:")
print(
    false_positives[
        [
            "content_id",
            "model_score",
            "search_volume",
            "days_since_last_update",
            "content_age_days",
            "trend_direction",
        ]
    ].head(3).to_string(index=False)
)

Top-50 model items: 50
Top-50 false positives: 31

False-positive rate within top 50:
0.620

Sample false positives:
          content_id  model_score  search_volume  days_since_last_update  content_age_days trend_direction
content_24b8d8860ad9     0.788589          880.0                      98               146            flat
content_45b7ba1087e0     0.778532          110.0                      98               146          stable
content_881635edbf8b     0.778207          110.0                      98               146             new


In [5]:
# Get transformed feature names from the fitted preprocessing pipeline.
feature_names = model.named_steps[
    "preprocessor"
].get_feature_names_out()

coefficients = model.named_steps[
    "classifier"
].coef_[0]

feature_importance = pd.DataFrame(
    {
        "feature": feature_names,
        "coefficient": coefficients,
        "absolute_coefficient": np.abs(coefficients),
    }
).sort_values(
    "absolute_coefficient",
    ascending=False
)

print("Top 10 model coefficients:")
print(
    feature_importance[
        ["feature", "coefficient"]
    ].head(10).to_string(index=False)
)

Top 10 model coefficients:
                                 feature  coefficient
        categorical__freshness_tier_181+    -0.876035
   categorical__main_intent_navigational    -0.543652
           categorical__age_tier_181-365    -0.333909
         numeric__days_since_last_update     0.329444
               numeric__content_age_days    -0.326736
categorical__content_type_feedly article    -0.272448
       categorical__freshness_tier_31-90     0.265049
      categorical__freshness_tier_91-180     0.252397
      categorical__word_count_tier_<1000    -0.247402
  categorical__main_intent_informational     0.233871


In [9]:
print("\nMODEL ERROR SUMMARY")
print("-------------------")

print(f"Test items: {len(test_results)}")
print(f"Top-50 false positives: {len(false_positives)}")
print(f"Top-50 precision: {model_p50:.3f}")

print("\nTop 3 model features by absolute coefficient:")

for _, row in feature_importance.head(3).iterrows():
    direction = "positive" if row["coefficient"] > 0 else "negative"

    print(
        f"- {row['feature']}: "
        f"{row['coefficient']:.3f} ({direction})"
    )


MODEL ERROR SUMMARY
-------------------
Test items: 6163
Top-50 false positives: 31
Top-50 precision: 0.380

Top 3 model features by absolute coefficient:
- categorical__freshness_tier_181+: -0.876 (negative)
- categorical__main_intent_navigational: -0.544 (negative)
- categorical__age_tier_181-365: -0.334 (negative)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.